In [1]:
import json
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
MODEL_ID       = "google/gemma-4-E4B-it"
OUTPUT_FILE    = Path("../data/translations_output_baseline.jsonl")
START          = 0      # first dataset index to process
NUM            = 8      # how many records to process
MAX_NEW_TOKENS = 2048   # generous limit for long solutions
BATCH_SIZE     = 2      # problem + solution → 1 batch; increase if VRAM allows
FEW_SHOT       = True

In [3]:
TRANSLATE_SYSTEM_PROMPT = """You are an expert mathematical translator specializing in English to Polish translation for olympiad and academic mathematics.

STRICT RULES:
1. Translate ALL natural language text from English to Polish
2. Keep ALL LaTeX commands, formulas, and math expressions EXACTLY as they are — do not modify anything inside $...$ or $$...$$
3. Keep variable names, labels, and point names unchanged (e.g. $A$, $B$, $ABC$, $f(x)$)
4. Output ONLY the translated text — no explanations, no comments, no preamble
5. Preserve all formatting, newlines, and structure from the original
6. Use correct Polish grammatical forms — pay special attention to:
   - noun declension (e.g. "trójkąt" → "trójkąta", "trójkątowi", "trójkącie")
   - adjective agreement (e.g. "prostokątny" must agree in gender/case with its noun)
   - verb conjugation (e.g. "udowodnij", "wyznacz", "oblicz", "pokaż, że")
   - preposition + case agreement (e.g. "dla trójkąta", "w okręgu", "na prostej")
   - correct mathematical terminology translation (e.g. calculus -> analiza matematyczna)
"""

TRANSLATE_PROBLEM_PROMPT = (
    "Translate the following math problem from English to Polish.\n"
    "Output ONLY the Polish translation, nothing else.\n\n"
    "TEXT TO TRANSLATE:\n{text}"
)

TRANSLATE_SOLUTION_PROMPT = (
    "Translate the following math solution from English to Polish.\n"
    "Output ONLY the Polish translation, nothing else.\n"
    "Preserve all mathematical steps, formulas, and logical structure.\n\n"
    "TEXT TO TRANSLATE:\n{text}"
)

system_message = [
    {
        "role": "system",
        "content": TRANSLATE_SYSTEM_PROMPT
    }
]

example_problems = [
    {
        "role": "user",
        "content": TRANSLATE_PROBLEM_PROMPT.format(text="\nProblem 1. Find all prime numbers $p$ for which there exist positive integers $x, y$ and $z$ such that the number\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z\n$$\n\nis a product of exactly three distinct prime numbers.\n")
    },
    {
        "role": "assistant",
        "content": "Zadanie 1. Znajdź wszystkie liczby pierwsze $p$, dla których istnieją dodatnie liczby całkowite $x, y$ i $z$ takie, że liczba\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z\n$$\n\njest iloczynem dokładnie trzech różnych liczb pierwszych."
    },
    {
        "role": "user",
        "content": TRANSLATE_PROBLEM_PROMPT.format(text="\nProblem 3. Triangle $A B C$ is such that $A B<A C$. The perpendicular bisector of side $B C$ intersects lines $A B$ and $A C$ at points $P$ and $Q$, respectively. Let $H$ be the orthocentre of triangle $A B C$, and let $M$ and $N$ be the midpoints of segments $B C$ and $P Q$, respectively. Prove that lines $H M$ and $A N$ meet on the circumcircle of $A B C$.\n")
    },
    {
        "role": "assistant",
        "content": "Zadanie 3. Trójkąt $A B C$ jest taki, że $A B < A C$. Symetralna boku $B C$ przecina proste $A B$ i $A C$ odpowiednio w punktach $P$ i $Q$. Niech $H$ będzie ortocentrum trójkąta $A B C$, a $M$ i $N$ — środkami odcinków $B C$ i $P Q$. Udowodnij, że proste $H M$ i $A N$ przecinają się na okręgu opisanym na trójkącie $A B C$."
    },
]

example_solutions = [
    {
        "role": "user",
        "content": TRANSLATE_SOLUTION_PROMPT.format(text="\nSolution. Let $A=x^{p}+y^{p}+z^{p}-x-y-z$. For $p=2$, we take $x=y=4$ and $z=3$. Then $A=30=2 \\cdot 3 \\cdot 5$. For $p=3$ we can take $x=3$ and $y=2$ and $z=1$. Then again $A=30=2 \\cdot 3 \\cdot 5$. For $p=5$ we can take $x=2$ and $y=1$ and $z=1$. Again $A=30=2 \\cdot 3 \\cdot 5$.\n\nAssume now that $p \\geqslant 7$. Working modulo 2 and modulo 3 we see that $A$ is divisible by both 2 and 3. Moreover, by Fermat's Little Theorem, we have\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z \\equiv x+y+z-x-y-z=0 \\bmod p \\text {. }\n$$\n\nTherefore, by the given condition, we have to solve the equation\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z=6 p\n$$\n\nIf one of the numbers $x, y$ and $z$ is bigger than or equal to 2 , let's say $x \\geqslant 2$, then\n\n$$\n6 p \\geqslant x^{p}-x=x\\left(x^{p-1}-1\\right) \\geqslant 2\\left(2^{p-1}-1\\right)=2^{p}-2\n$$\n\nIt is easy to check by induction that $2^{n}-2>6 n$ for all natural numbers $n \\geqslant 6$. This contradiction shows that there are no more values of $p$ which satisfy the required property.\n\nRemark. There are a couple of other ways to prove that $2^{p}-2>6 p$ for $p \\geqslant 7$. For example, we can use the Binomial Theorem as follows:\n\n$$\n2^{p}-2 \\geqslant 1+p+\\frac{p(p-1)}{2}+\\frac{p(p-1)(p-2)}{6}-2 \\geqslant 1+p+3 p+5 p-2>6 p\n$$\n\nWe can also use Bernoulli's Inequality as follows:\n\n$$\n2^{p}-2=8(1+1)^{p-3}-2 \\geqslant 8(1+(p-3))-2=8 p-18>6 p\n$$\n\nThe last inequality is true for $p \\geqslant 11$. For $p=7$ we can see directly that $2^{p}-2>6 p$.\n\nOne can also use calculus to show that $f(x)=2^{x}-6 x$ is increasing for $x \\geqslant 5$.\n")
    },
    {
        "role": "assistant",
        "content": "Rozwiązanie. Niech $A=x^{p}+y^{p}+z^{p}-x-y-z$. Dla $p=2$ bierzemy $x=y=4$ i $z=3$. Wtedy $A=30=2 \\cdot 3 \\cdot 5$. Dla $p=3$ możemy wziąć $x=3$ i $y=2$ i $z=1$. Znowu $A=30=2 \\cdot 3 \\cdot 5$. Dla $p=5$ możemy wziąć $x=2$ i $y=1$ i $z=1$. Znowu $A=30=2 \\cdot 3 \\cdot 5$.\n\nZałóżmy teraz, że $p \\geqslant 7$. Pracując modulo 2 i modulo 3 widzimy, że $A$ jest dzielny przez 2 i 3. Ponadto, z Małego Twierdzenia Fermata mamy\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z \\equiv x+y+z-x-y-z=0 \\bmod p \\text {. }\n$$\n\nDlatego, z danego warunku, musimy rozwiązać równanie\n\n$$\nx^{p}+y^{p}+z^{p}-x-y-z=6 p\n$$\n\nJeśli jedna z liczb $x, y$ i $z$ jest większa lub równa 2, powiedzmy $x \\geqslant 2$, to\n\n$$\n6 p \\geqslant x^{p}-x=x\\left(x^{p-1}-1\\right) \\geqslant 2\\left(2^{p-1}-1\\right)=2^{p}-2\n$$\n\nŁatwo sprawdzić indukcyjnie, że $2^{n}-2>6 n$ dla wszystkich liczb naturalnych $n \\geqslant 6$. Ten sprzeczność pokazuje, że nie ma więcej wartości $p$, które spełniają wymagane własność.\n\nUwaga. Istnieją parę innych sposobów, aby udowodnić, że $2^{p}-2>6 p$ dla $p \\geqslant 7$. Na przykład, możemy użyć Twierdzenia Binomialnego w następujący sposób:\n\n$$\n2^{p}-2 \\geqslant 1+p+\\frac{p(p-1)}{2}+\\frac{p(p-1)(p-2)}{6}-2 \\geqslant 1+p+3 p+5 p-2>6 p\n$$\n\nMożemy również użyć Nierówności Bernoulliego w następujący sposób:\n\n$$\n2^{p}-2=8(1+1)^{p-3}-2 \\geqslant 8(1+(p-3))-2=8 p-18>6 p\n$$\n\nOstatnia nierówność jest prawdziwa dla $p \\geqslant 11$. Dla $p=7$ możemy zobaczyć bezpośrednio, że $2^{p}-2>6 p$.\n\nMożna również użyć rachunku, aby pokazać, że $f(x)=2^{x}-6 x$ jest rosnący dla $x \\geqslant 5$."
    },
    {
        "role": "user",
        "content": TRANSLATE_SOLUTION_PROMPT.format(text="\nSolution. If we colour all the cells along all edges of the board together with the entire middle row except the second and the last-but-one cell, the condition is satisfied and there are 302 black cells. The figure below exhibits this colouring for the $5 \\times 8$ case.\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_fe2687448771fc19bd4eg-5.jpg?height=320&width=493&top_left_y=629&top_left_x=816)\n\nWe can cover the table by one fragment like the first one on the figure below, 24 fragments like the middle one, and one fragment like the third one.\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_fe2687448771fc19bd4eg-5.jpg?height=317&width=737&top_left_y=1132&top_left_x=694)\n\nIn each fragment, among the cells with the same letter, there are at most two coloured black, so the total number of coloured cells is at most $(5+24 \\cdot 6+1) \\cdot 2+2=302$.\n")
    },
    {
        "role": "assistant",
        "content": "Rozwiązanie. Jeśli pokolorujemy wszystkie komórki wzdłuż wszystkich krawędzi planszy oraz cały środkowy rząd z wyjątkiem drugiej i przedostatniej komórki, warunek jest spełniony i mamy 302 czarne komórki. Poniższy rysunek przedstawia to pokolorowanie dla przypadku $5 \\times 8$.\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_fe2687448771fc19bd4eg-5.jpg?height=320&width=493&top_left_y=629&top_left_x=816)\n\nMożemy pokryć tabelę jednym fragmentem jak pierwszy na poniższym rysunku, 24 fragmentami jak ten środkowy, i jednym fragmentem jak trzeci.\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_fe2687448771fc19bd4eg-5.jpg?height=317&width=737&top_left_y=1132&top_left_x=694)\n\nW każdym fragmencie, wśród komórek o tej samej literze, jest co najwyżej dwie pokolorowane na czarno, więc łączna liczba pokolorowanych komórek jest co najwyżej $(5+24 \\cdot 6+1) \\cdot 2+2=302$."
    },
]

base_problems = system_message
base_solutions = system_message

if FEW_SHOT:
    base_problems += example_problems
    base_solutions += example_solutions   

In [4]:
ds = load_dataset("AI-MO/NuminaMath-1.5")
train = ds["train"]
print(f"Dataset size: {len(train)} records")

Dataset size: 896215 records


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Left-pad so all sequences in a batch are right-aligned — required for generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model ready.")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Model ready.


In [6]:
def call_llm_batch(items: list[dict]) -> list[tuple[str, str]]:
    """
    Batch inference over multiple dataset items.

    Each item:
        {
            "problem": str,
            "solution": str
        }

    Returns:
        [(problem_pl, solution_pl), ...]
    """

    texts = []

    # flatten: 2 prompts per item (problem + solution)
    for item in items:
        texts.append(
            tokenizer.apply_chat_template(base_problems +
                [{"role": "user", "content": f"{TRANSLATE_PROBLEM_PROMPT.format(text=item['problem'])}"}],
                tokenize=False,
                add_generation_prompt=True,
            )
        )
        texts.append(
            tokenizer.apply_chat_template(base_solutions +
                [{"role": "user", "content": f"{TRANSLATE_SOLUTION_PROMPT.format(text=item['solution'])}"}],
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    results = []
    for i in range(0, len(outputs), 2):
        problem_pl = tokenizer.decode(outputs[i][input_len:], skip_special_tokens=True)
        solution_pl = tokenizer.decode(outputs[i + 1][input_len:], skip_special_tokens=True)
        results.append((problem_pl, solution_pl))

    return results

In [7]:
def get_processed_ids(output_file: Path) -> set:
    if not output_file.exists():
        return set()
    with open(output_file, "r", encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}


def save_translation(output_file: Path, idx: int, task: dict, problem_pl: str, solution_pl: str) -> None:
    record = {
        "id": idx,
        "problem_en": task["problem"],
        "problem_pl": problem_pl,
        "solution_en": task["solution"],
        "solution_pl": solution_pl,
    }
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [8]:
%%time
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

processed = get_processed_ids(OUTPUT_FILE)
print(f"Already processed: {len(processed)} records")

end   = min(START + NUM, len(train))
total = end - START

for i in range(START, end, BATCH_SIZE):

    batch_items = []
    batch_indices = []

    for j in range(i, min(i + BATCH_SIZE, end)):
        if j in processed:
            print(f"Skipping {j} (already done)")
            continue

        task = train[j]
        batch_items.append(task)
        batch_indices.append(j)

    if not batch_items:
        continue

    try:
        results = call_llm_batch(batch_items)

        for idx, task, (problem_pl, solution_pl) in zip(batch_indices, batch_items, results):
            save_translation(
                OUTPUT_FILE,
                idx,
                task,
                problem_pl,
                solution_pl,
            )

        print(f"[batch {i}-{i+BATCH_SIZE}] ✓ saved")

    except Exception as e:
        print(f"[batch {i}-{i+BATCH_SIZE}] ✗ Error: {e}")
        continue

Already processed: 0 records
[batch 0-2] ✓ saved
[batch 2-4] ✓ saved
[batch 4-6] ✓ saved
[batch 6-8] ✓ saved
CPU times: user 5min 46s, sys: 4.69 s, total: 5min 50s
Wall time: 5min 54s
